# Lab 1 — HDFS: estrutura, upload e replicação

## Objetivo

Este laboratório tem como objetivo compreender a organização dos dados em camadas e o conceito de replicação utilizado pelo HDFS.

Foi utilizada a rota sem privilégios administrativos. Nessa alternativa, as pastas locais simulam a estrutura lógica de um Data Lake, com as camadas Raw, Bronze, Silver e Gold. A replicação é demonstrada por meio de cópias locais, mas não corresponde a uma replicação distribuída real, pois os arquivos permanecem no mesmo computador.

In [9]:
from pathlib import Path
import shutil
import hashlib
import pandas as pd

# O VS Code normalmente inicia o notebook na raiz do projeto.
pasta_projeto = Path.cwd().resolve()

# Caso o diretório atual não seja a raiz, procura uma pasta que contenha dados/raw.
if not (pasta_projeto / "dados" / "raw").exists():
    for pasta_pai in pasta_projeto.parents:
        if (pasta_pai / "dados" / "raw").exists():
            pasta_projeto = pasta_pai
            break

pasta_raw = pasta_projeto / "dados" / "raw"
pasta_bronze = pasta_projeto / "dados" / "bronze"
pasta_silver = pasta_projeto / "dados" / "silver"
pasta_gold = pasta_projeto / "dados" / "gold"

print("Pasta do projeto:", pasta_projeto)
print("Pasta Raw:", pasta_raw)

assert pasta_raw.exists(), "A pasta dados/raw não foi encontrada."

Pasta do projeto: C:\BigData\bigdata-curso-gabriel
Pasta Raw: C:\BigData\bigdata-curso-gabriel\dados\raw


In [10]:
# Subpastas específicas da camada Raw
subpastas_raw = {
    "customers": pasta_raw / "customers",
    "transactions": pasta_raw / "transactions",
    "fraud_labels": pasta_raw / "fraud_labels"
}

# Criação das camadas e subpastas
for pasta in [
    pasta_raw,
    pasta_bronze,
    pasta_silver,
    pasta_gold,
    *subpastas_raw.values()
]:
    pasta.mkdir(parents=True, exist_ok=True)

# Arquivos atualmente localizados diretamente em dados/raw
arquivos_origem = {
    "customers": pasta_raw / "customers_synthetic.csv",
    "transactions": pasta_raw / "transactions_synthetic.csv",
    "fraud_labels": pasta_raw / "fraud_labels.csv"
}

# Copia cada arquivo para sua subpasta, sem apagar o original
for nome, origem in arquivos_origem.items():
    destino = subpastas_raw[nome] / origem.name

    if not origem.exists() and destino.exists():
        print(f"Já organizado: {destino.relative_to(pasta_projeto)}")
    elif origem.exists():
        shutil.copy2(origem, destino)
        print(f"Copiado: {origem.name} → {destino.relative_to(pasta_projeto)}")
    else:
        print(f"ATENÇÃO: arquivo não encontrado — {origem.name}")

Copiado: customers_synthetic.csv → dados\raw\customers\customers_synthetic.csv
Copiado: transactions_synthetic.csv → dados\raw\transactions\transactions_synthetic.csv
Copiado: fraud_labels.csv → dados\raw\fraud_labels\fraud_labels.csv


In [11]:
print("ESTRUTURA DE DADOS\n")

for camada in ["raw", "bronze", "silver", "gold"]:
    pasta_camada = pasta_projeto / "dados" / camada
    print(f"{camada.upper()}/")

    arquivos = sorted(
        arquivo
        for arquivo in pasta_camada.rglob("*")
        if arquivo.is_file()
    )

    if arquivos:
        for arquivo in arquivos:
            caminho_relativo = arquivo.relative_to(pasta_camada)
            tamanho_mb = arquivo.stat().st_size / (1024 ** 2)
            print(f"  {caminho_relativo} — {tamanho_mb:.2f} MB")
    else:
        print("  Pasta vazia")

    print()

ESTRUTURA DE DADOS

RAW/
  customers\customers_synthetic.csv — 0.80 MB
  customers_synthetic.csv — 0.80 MB
  fraud_labels\fraud_labels.csv — 0.21 MB
  fraud_labels.csv — 0.21 MB
  transactions\transactions_synthetic.csv — 7.76 MB
  transactions_synthetic.csv — 7.76 MB

BRONZE/
  Pasta vazia

SILVER/
  Pasta vazia

GOLD/
  Pasta vazia



In [12]:
arquivos_dados = {
    "Clientes": subpastas_raw["customers"] / "customers_synthetic.csv",
    "Transações": subpastas_raw["transactions"] / "transactions_synthetic.csv",
    "Rótulos de fraude": subpastas_raw["fraud_labels"] / "fraud_labels.csv"
}

resultado_contagem = []

for base, caminho in arquivos_dados.items():
    df = pd.read_csv(caminho)

    resultado_contagem.append({
        "Base": base,
        "Arquivo": caminho.name,
        "Linhas": len(df),
        "Colunas": len(df.columns)
    })

contagem_df = pd.DataFrame(resultado_contagem)
contagem_df

,Base,Arquivo,Linhas,Colunas
0,Clientes,customers_synthetic.csv,9993,7
1,Transações,transactions_synthetic.csv,100000,8
2,Rótulos de fraude,fraud_labels.csv,4833,5


In [13]:
quantidades_esperadas = {
    "Clientes": 9_993,
    "Transações": 100_000,
    "Rótulos de fraude": 4_833
}

contagem_df["Linhas esperadas"] = contagem_df["Base"].map(
    quantidades_esperadas
)

contagem_df["Validação"] = (
    contagem_df["Linhas"] == contagem_df["Linhas esperadas"]
).map({
    True: "OK",
    False: "DIVERGENTE"
})

contagem_df

,Base,Arquivo,Linhas,Colunas,Linhas esperadas,Validação
0,Clientes,customers_synthetic.csv,9993,7,9993,OK
1,Transações,transactions_synthetic.csv,100000,8,100000,OK
2,Rótulos de fraude,fraud_labels.csv,4833,5,4833,OK


In [14]:
arquivo_transacoes = (
    subpastas_raw["transactions"] / "transactions_synthetic.csv"
)

pasta_replica = pasta_projeto / "dados" / "_replica_demo"
pasta_replica.mkdir(parents=True, exist_ok=True)

copias = []

for numero in range(1, 4):
    destino = pasta_replica / f"copia_{numero}.csv"
    shutil.copy2(arquivo_transacoes, destino)
    copias.append(destino)

print("Replicação local simulada:")

for arquivo in copias:
    tamanho_mb = arquivo.stat().st_size / (1024 ** 2)
    print(f"{arquivo.name}: {tamanho_mb:.2f} MB")

Replicação local simulada:
copia_1.csv: 7.76 MB
copia_2.csv: 7.76 MB
copia_3.csv: 7.76 MB


In [15]:
def calcular_hash(caminho, tamanho_bloco=1024 * 1024):
    hash_arquivo = hashlib.sha256()

    with open(caminho, "rb") as arquivo:
        while bloco := arquivo.read(tamanho_bloco):
            hash_arquivo.update(bloco)

    return hash_arquivo.hexdigest()


arquivos_verificados = [arquivo_transacoes, *copias]

resultado_replicacao = pd.DataFrame({
    "Arquivo": [arquivo.name for arquivo in arquivos_verificados],
    "Tamanho em bytes": [
        arquivo.stat().st_size for arquivo in arquivos_verificados
    ],
    "Hash SHA-256": [
        calcular_hash(arquivo) for arquivo in arquivos_verificados
    ]
})

resultado_replicacao["Cópia idêntica"] = (
    resultado_replicacao["Hash SHA-256"]
    == resultado_replicacao.loc[0, "Hash SHA-256"]
).map({
    True: "SIM",
    False: "NÃO"
})

resultado_replicacao

,Arquivo,Tamanho em bytes,Hash SHA-256,Cópia idêntica
0,transactions_synthetic.csv,8139648,55f45648d22b684e242f0e097d2f8d793638abdda0987b...,SIM
1,copia_1.csv,8139648,55f45648d22b684e242f0e097d2f8d793638abdda0987b...,SIM
2,copia_2.csv,8139648,55f45648d22b684e242f0e097d2f8d793638abdda0987b...,SIM
3,copia_3.csv,8139648,55f45648d22b684e242f0e097d2f8d793638abdda0987b...,SIM


## Conclusão

A estrutura local foi organizada nas camadas Raw, Bronze, Silver e Gold. As três bases fornecidas foram armazenadas em subpastas específicas da camada Raw e tiveram suas quantidades de registros verificadas.

A simulação produziu três cópias idênticas da base de transações. A igualdade foi confirmada pelo tamanho dos arquivos e pelo hash SHA-256. Essa atividade representa conceitualmente o fator de replicação do HDFS, utilizado para aumentar a disponibilidade e a tolerância a falhas.